# Complete Phase-dependent GMM models with LOOCV

In [1]:
# load packages and data
import numpy as np
import pickle
import math
import time
import sys
module_paths = ['']
for path in module_paths :
    if path not in sys.path:
        sys.path.append(path)
from utils_load_PHloc import datasets_of_interest, injected_datasets_of_interest, levels_of_interest
import matplotlib.pyplot as plt
import pomegranate
import sklearn
import torch
from pomegranate.gmm import GeneralMixtureModel
from pomegranate.distributions import *
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pandas as pd 
from sklearn.utils import resample

PH_folder = '' 
anatomy = 'knee'
filepath = PH_folder + 'PH_all_{}.pkl'.format(anatomy)
PH_all_datasets = pickle.load(open(filepath, 'rb'))

# threshold value
THR = .5
# generate a new dictionary for truncated datasets
truncated_PH_all_datasets = {}
for i in datasets_of_interest:
    diagram = PH_all_datasets[i]
    truncated_PH_all_datasets[i] = diagram[diagram[:,1] >= diagram[:,0] + THR]
    
names = [injected_datasets_of_interest[i]+"_"+[str(x) for x in datasets_of_interest][i] for i in range(27)]

labels = ['CTRL_0%(1)', 'CTRL_0%(2)', 'CTRL_0%(3)', 'CTRL_0%(4)', 
          'U937_1%(1)', 'U937_1%(2)', 'U937_7%', 'U937_8%', 'U937_10%(1)', 'U937_10%(2)', 'U937_10%(3)', 
          'HL60_23%', 'HL60_25%(1)', 'HL60_25%(2)', 
          'P1_10%', 'P1_40%', 'P1_44%', 'P1_51%', 'P1_60%', 'P1_76%', 
          'P2_59%', 'P2_88%', 'P2_90%', 
          'MNC_53%', 'MNC_67%', 'MNC_75%', 'MNC_86%']

phases_of_interest = [0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,2,
         2,2,2,
         2,2,2,2]

name_phase = [labels[i]+" [Phase "+str(phases_of_interest[i])+"]" for i in range(27)]

# Binning parameters
XLIMS = np.array([[-15,0],[-10,10],[0,20]])
YLIMS = np.array([[-8,7],[-5,15],[0,20]])
NB_BINS_PER_SIDE = 100

pds = [truncated_PH_all_datasets[i] for i in datasets_of_interest]

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


In [2]:
# function that extract quadrant data according to the name
def extract_quadrant(quadrant_name):
    # quadrant names in the form of ph1_ne etc
    # Step 1: get dimension
    dim = int(quadrant_name[2])
    pds_dim = []
    for i in range(27):
        current_pd = pds[i]
        pds_dim.append(current_pd[current_pd[:,2]==dim,:2])
    # Step 2: extract based on orientation
    out_pds = []
    if quadrant_name[4:]=="ne":
        for i in range(27):
            pd = pds_dim[i]
            x = pd[:,0]
            y = pd[:,1]
            out_pds.append(pd[(x>0)*(y>0),:])
    elif quadrant_name[4:]=="sw":
        for i in range(27):
            pd = pds_dim[i]
            x = pd[:,0]
            y = pd[:,1]
            out_pds.append(pd[(x<0)*(y<0),:])
    elif quadrant_name[4:]=="nw":
        for i in range(27):
            pd = pds_dim[i]
            x = pd[:,0]
            y = pd[:,1]
            out_pds.append(pd[(x<0)*(y>0),:])
    return out_pds

# dictionary for number of components for each phase model
num_component_dict = {"ph1_ne":[5,6,6], "ph0_sw":[4,5,5], "ph1_nw":[3,4,5], "ph1_sw":[5,5,7], "ph2_ne":[4,4,5]}

In [3]:
ph1_ne = extract_quadrant("ph1_ne")
ph0_sw = extract_quadrant("ph0_sw")
ph1_nw = extract_quadrant("ph1_nw")
ph1_sw = extract_quadrant("ph1_sw")
ph2_ne = extract_quadrant("ph2_ne")
print(f"Sizes in ph1_ne: {[x.shape[0] for x in ph1_ne]}")
print(f"Sizes in ph0_sw: {[x.shape[0] for x in ph0_sw]}")
print(f"Sizes in ph1_nw: {[x.shape[0] for x in ph1_nw]}")
print(f"Sizes in ph1_sw: {[x.shape[0] for x in ph1_sw]}")
print(f"Sizes in ph2_ne: {[x.shape[0] for x in ph2_ne]}")

Sizes in ph1_ne: [15254, 10529, 14935, 12883, 7483, 10886, 21364, 15168, 20547, 13022, 13257, 12717, 14110, 14010, 16710, 12066, 14356, 11077, 9768, 8800, 12883, 9324, 6573, 11560, 10577, 10170, 11633]
Sizes in ph0_sw: [13940, 9202, 12838, 11514, 6376, 8818, 15368, 11217, 14819, 9920, 11126, 10393, 9837, 9701, 12462, 8955, 10483, 8453, 8210, 6566, 9302, 6448, 5054, 9622, 7535, 7838, 7846]
Sizes in ph1_nw: [12759, 7935, 10387, 10729, 5259, 8473, 14626, 10495, 14366, 7949, 9096, 8775, 8537, 9854, 11567, 6525, 6855, 4629, 5518, 5104, 4595, 3458, 2801, 8762, 3983, 4270, 3905]
Sizes in ph1_sw: [2629, 1490, 2192, 2524, 902, 1924, 2466, 2386, 2765, 1210, 1637, 1730, 1884, 1424, 2747, 1210, 1371, 1176, 1863, 1101, 1327, 959, 700, 1454, 1009, 1418, 714]
Sizes in ph2_ne: [9904, 6305, 8434, 8314, 3781, 6693, 11687, 8731, 11625, 6367, 7215, 7368, 7667, 7846, 9231, 6114, 6475, 4886, 4977, 4441, 5392, 3629, 2899, 6142, 4088, 4794, 4656]


# Carry out LOOCV in three ways:

- Simple LOOCV on 27 samples only
- LOOCV and bootstrap on each train set SDPH diagram but not test diagram
- LOOCV and bootstrap from pooled diagrams but not test diagram

The predictions can be made in several ways, based on:
- loglikelihood
- Hellinger distance
- KL divergence

In [4]:
def kl_divergence(model1, model2, dim):
    x = np.arange(XLIMS[dim][0],XLIMS[dim][1],.1)
    y = np.arange(YLIMS[dim][0],YLIMS[dim][1],.1)
    xx,yy = np.meshgrid(x,y)
    x_ = np.array(list(zip(xx.flatten(), yy.flatten())))

    eps = 1e-15
    p1 = model1.probability(x_).reshape(len(x),len(y)) + eps
    p2 = model2.probability(x_).reshape(len(x),len(y)) + eps
     
    p1 = p1/np.sum(p1)
    p2 = p2/np.sum(p2)
    
    return np.sum(p1*(np.log(p1/p2))) 

# Hellinger distance
def Hellinger(model1, model2, dim):
    x = np.arange(XLIMS[dim][0],XLIMS[dim][1],.1)
    y = np.arange(YLIMS[dim][0],YLIMS[dim][1],.1)
    xx,yy = np.meshgrid(x,y)
    x_ = np.array(list(zip(xx.flatten(), yy.flatten())))

    p1 = model1.probability(x_).reshape(len(x),len(y))
    p2 = model2.probability(x_).reshape(len(x),len(y))
     
    p1 = p1/np.sum(p1)
    p2 = p2/np.sum(p2)
    
    summation = np.sum(np.square(np.sqrt(p1)-np.sqrt(p2)))   
    
    return np.sqrt(summation)/np.sqrt(2)
  

In [5]:
# --- Parallelized LOOCV Function ---
def process_fold_simple(train_index, test_index, X, y, n_components, phases, dim, num_subjects):
    X_train_list = [X[ind] for ind in train_index] # Keep as list of arrays
    X_test_single = np.array(X[test_index[0]]) # Test data for one subject
    y_train = y[train_index]
    y_test = y[test_index][0] # y_test is a single value

    # Prepare data for each phase model
    phase_data = {}
    for p in phases:
        # Filter X_train_list based on y_train
        current_phase_samples = [X_train_list[i] for i in range(len(X_train_list)) if y_train[i] == p]
        # print(f"length for phase {p}: {len(current_phase_samples)}")
        if current_phase_samples:
            phase_data[p] = np.vstack(current_phase_samples)
        else:
            phase_data[p] = np.array([]) 
            print("empty")

    # Fit GMMs for each phase
    models = {}
    for p_idx, p in enumerate(phases):
        if phase_data[p].shape[0] > 0:
            try:
                model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=phase_data[p]
                )
                model.fit(
                    X=phase_data[p],
                    weights=phase_data[p][:,1] - phase_data[p][:,0], 
                    stop_threshold=.001,
                    verbose=False # Suppress verbose output during fitting
                )
                models[p] = model
            except Exception as e:
                # Handle cases where GMM fitting might fail (e.g., too few samples)
                # print(f"Warning: GMM fitting failed for phase {p} in a fold: {e}")
                models[p] = None
        else:
            models[p] = None

    # --- Prediction via loglikelihood ---
    bics = []
    for p in phases:
        ll = np.sum(models[p].log_probability(X_test_single))
        bics.append(-2*ll+math.log(phase_data[p].shape[0])*6*(p+2))
    # based on which model minimizes bic
    y_pred_l = phases[np.argmin(bics)]

    # --- Prepare test models for Hellinger and KL-Divergence ---
    test_models = {}
    for p_idx, p in enumerate(phases):
        if X_test_single.shape[0] > 0:
            try:
                test_model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=X_test_single
                )
                test_model.fit(
                    X=X_test_single,
                    weights=X_test_single[:,1]-X_test_single[:,0], 
                    stop_threshold=.001,
                    verbose=False
                )
                test_models[p] = test_model
            except Exception as e:
                # print(f"Warning: Test GMM fitting failed for phase {p} in a fold: {e}")
                test_models[p] = None
        else:
            test_models[p] = None

    # --- Prediction via Hellinger distance ---
    hdist = []
    for p in phases:
        if models[p] and test_models[p]:
            hdist.append(Hellinger(models[p], test_models[p], dim))
        else:
            hdist.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_h = phases[np.argmin(hdist)] if not np.all(np.isinf(hdist)) else -1

    # --- Prediction via KL-divergence ---
    kldiv = []
    for p in phases:
        if models[p] and test_models[p]:
            kldiv.append(kl_divergence(models[p], test_models[p], dim))
        else:
            kldiv.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_k = phases[np.argmin(kldiv)] if not np.all(np.isinf(kldiv)) else -1

    return y_test, y_pred_l, y_pred_h, y_pred_k

In [6]:
# --- Main execution loop for multiple regions ---
regions_to_analyze = ["ph1_ne", "ph0_sw", "ph1_nw", "ph1_sw", "ph2_ne"]
all_results = {}

for region in regions_to_analyze:
    print(f"\n--- Processing Region: {region} ---")

    X = extract_quadrant(region)
    y = np.array([0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,2,
         2,2,2,
         2,2,2,2])
    n_components = num_component_dict[region]
    num_subjects = 27 
    loocv = LeaveOneOut()
    phases = [0, 1, 2] # class labels
    dim = int(region[2]) 

    # Parallel execution for the current region
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_fold_simple)(train_index, test_index, X, y, n_components, phases, dim, num_subjects)
        for i, (train_index, test_index) in enumerate(loocv.split(X))
    )

    # Unpack results for the current region
    y_test_true_region = [res[0] for res in fold_results]
    loglikelihood_pred_region = [res[1] for res in fold_results]
    hellinger_pred_region = [res[2] for res in fold_results]
    kl_divergence_pred_region = [res[3] for res in fold_results]        

    # --- Calculate and store metrics for the current region ---
    region_metrics = {}

    # Log-likelihood Metrics
    region_metrics['loglikelihood'] = {
        'accuracy': accuracy_score(y_test_true_region, loglikelihood_pred_region),
        'f1_macro': f1_score(y_test_true_region, loglikelihood_pred_region, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test_true_region, loglikelihood_pred_region, labels=phases),
        'classification_report': classification_report(y_test_true_region, loglikelihood_pred_region, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
    }

    # Hellinger Distance Metrics
    region_metrics['hellinger'] = {
        'accuracy': accuracy_score(y_test_true_region, hellinger_pred_region),
        'f1_macro': f1_score(y_test_true_region, hellinger_pred_region, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test_true_region, hellinger_pred_region, labels=phases),
        'classification_report': classification_report(y_test_true_region, hellinger_pred_region, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
    }
    
    # KL-Divergence Metrics
    region_metrics['kl_divergence'] = {
        'accuracy': accuracy_score(y_test_true_region, kl_divergence_pred_region),
        'f1_macro': f1_score(y_test_true_region, kl_divergence_pred_region, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test_true_region, kl_divergence_pred_region, labels=phases),
        'classification_report': classification_report(y_test_true_region, kl_divergence_pred_region, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
    }

    all_results[region] = region_metrics


# --- Display Results for All Regions ---
print("\n" + "="*50)
print("             OVERALL RESULTS SUMMARY             ")
print("="*50)

for region, metrics in all_results.items():
    print(f"\n### Region: {region} ###")

    for method, data in metrics.items():
        print(f"\n--- Method: {method.replace('_', ' ').title()} ---")
        print(f"Accuracy: {data['accuracy']:.4f}")
        print(f"F1-Macro Score: {data['f1_macro']:.4f}")

        if data['confusion_matrix'] is not None:
            print("\nConfusion Matrix:")
            cm_df = pd.DataFrame(data['confusion_matrix'],
                                 index=[f'True Phase {p}' for p in phases],
                                 columns=[f'Pred Phase {p}' for p in phases])
            print(cm_df)
        else:
            print("\nConfusion Matrix: Not available (no valid predictions)")

        if data['classification_report'] is not None:
            print("\nClassification Report:")
            # Convert classification report dict to a nice DataFrame for display
            report_df = pd.DataFrame(data['classification_report']).transpose()
            print(report_df)
        else:
            print("\nClassification Report: Not available (no valid predictions)")

    print("\n" + "-"*40)


--- Processing Region: ph1_ne ---

--- Processing Region: ph0_sw ---

--- Processing Region: ph1_nw ---

--- Processing Region: ph1_sw ---

--- Processing Region: ph2_ne ---

             OVERALL RESULTS SUMMARY             

### Region: ph1_ne ###

--- Method: Loglikelihood ---
Accuracy: 0.5556
F1-Macro Score: 0.3889

Confusion Matrix:
              Pred Phase 0  Pred Phase 1  Pred Phase 2
True Phase 0             0             4             0
True Phase 1             0            11             0
True Phase 2             1             7             4

Classification Report:
              precision    recall  f1-score    support
Phase 0        0.000000  0.000000  0.000000   4.000000
Phase 1        0.500000  1.000000  0.666667  11.000000
Phase 2        1.000000  0.333333  0.500000  12.000000
accuracy       0.555556  0.555556  0.555556   0.555556
macro avg      0.500000  0.444444  0.388889  27.000000
weighted avg   0.648148  0.555556  0.493827  27.000000

--- Method: Hellinger ---
Accu

### Method 2: training from bootstrap samples of each

In [7]:
# --- Parallelized LOOCV Function ---
def process_fold_bootstrap(train_index, test_index, X, y, n_components, phases, dim, num_subjects):
    X_train_list = [X[ind] for ind in train_index] # Keep as list of arrays
    X_test_single = np.array(X[test_index[0]]) # Test data for one subject
    y_train = y[train_index]
    y_test = y[test_index][0] # y_test is a single value

    # for each pd in X_train_list replace with a stack of 5 bootstrap samples
    train_list = []
    for train_pd in X_train_list:
        pds = []
        for iters in range(5):
            pds.append(resample(train_pd, n_samples=int(np.floor(train_pd.shape[0]/4)), replace=True))
        train_list.append(np.vstack(pds))
    X_train_list = train_list
    # Prepare data for each phase model
    phase_data = {}
    for p in phases:
        # Filter X_train_list based on y_train
        current_phase_samples = [X_train_list[i] for i in range(len(X_train_list)) if y_train[i] == p]
        if current_phase_samples:
            phase_data[p] = np.vstack(current_phase_samples)
        else:
            phase_data[p] = np.array([]) # Empty array if no data for this phase

    # Fit GMMs for each phase
    models = {}
    for p_idx, p in enumerate(phases):
        if phase_data[p].shape[0] > 0:
            try:
                model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=phase_data[p]
                )
                model.fit(
                    X=phase_data[p],
                    weights=phase_data[p][:,1]-phase_data[p][:,0], 
                    stop_threshold=.001,
                    verbose=False # Suppress verbose output during fitting
                )
                models[p] = model
            except Exception as e:
                # Handle cases where GMM fitting might fail (e.g., too few samples)
                # print(f"Warning: GMM fitting failed for phase {p} in a fold: {e}")
                models[p] = None
        else:
            models[p] = None

    # --- Prediction via loglikelihood ---
    bics = []
    for p in phases:
        ll = np.sum(models[p].log_probability(X_test_single))
        bics.append(-2*ll+math.log(phase_data[p].shape[0])*6*(p+2))
    # based on which model minimizes bic
    y_pred_l = phases[np.argmin(bics)]

    # --- Prepare test models for Hellinger and KL-Divergence ---
    test_models = {}
    for p_idx, p in enumerate(phases):
        if X_test_single.shape[0] > 0:
            try:
                test_model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=X_test_single
                )
                test_model.fit(
                    X=X_test_single,
                    weights=X_test_single[:,1]-X_test_single[:,0], 
                    stop_threshold=.001,
                    verbose=False
                )
                test_models[p] = test_model
            except Exception as e:
                # print(f"Warning: Test GMM fitting failed for phase {p} in a fold: {e}")
                test_models[p] = None
        else:
            test_models[p] = None

    # --- Prediction via Hellinger distance ---
    hdist = []
    for p in phases:
        if models[p] and test_models[p]:
            hdist.append(Hellinger(models[p], test_models[p], dim))
        else:
            hdist.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_h = phases[np.argmin(hdist)] if not np.all(np.isinf(hdist)) else -1

    # --- Prediction via KL-divergence ---
    kldiv = []
    for p in phases:
        if models[p] and test_models[p]:
            kldiv.append(kl_divergence(models[p], test_models[p], dim))
        else:
            kldiv.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_k = phases[np.argmin(kldiv)] if not np.all(np.isinf(kldiv)) else -1

    return y_test, y_pred_l, y_pred_h, y_pred_k

In [8]:
# --- Main execution loop for multiple regions ---
regions_to_analyze = ["ph1_ne", "ph0_sw", "ph1_nw", "ph1_sw", "ph2_ne"]
all_results = {}

for region in regions_to_analyze:
    print(f"\n--- Processing Region: {region} ---")

    X = extract_quadrant(region)
    y = np.array([0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,2,
         2,2,2,
         2,2,2,2])
    n_components = num_component_dict[region]
    num_subjects = 27 
    loocv = LeaveOneOut()
    phases = [0, 1, 2] # class labels
    dim = int(region[2]) 

    # Parallel execution for the current region
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_fold_bootstrap)(train_index, test_index, X, y, n_components, phases, dim, num_subjects)
        for i, (train_index, test_index) in enumerate(loocv.split(X))
    )

    # Unpack results for the current region
    y_test_true_region = [res[0] for res in fold_results]
    loglikelihood_pred_region = [res[1] for res in fold_results]
    hellinger_pred_region = [res[2] for res in fold_results]
    kl_divergence_pred_region = [res[3] for res in fold_results]

    # Filter out unclassifiable predictions for metric calculation (predictions of -1)
    # This ensures metrics are only calculated on cases where a prediction was actually made
    filtered_y_test_ll = [y_t for y_t, y_p in zip(y_test_true_region, loglikelihood_pred_region) if y_p != -1]
    filtered_y_pred_ll = [y_p for y_p in loglikelihood_pred_region if y_p != -1]
    if len(filtered_y_pred_ll)!=27:
        print("missing prediction from loglikelihood")
        
    filtered_y_test_h = [y_t for y_t, y_p in zip(y_test_true_region, hellinger_pred_region) if y_p != -1]
    filtered_y_pred_h = [y_p for y_p in hellinger_pred_region if y_p != -1]
    if len(filtered_y_pred_h)!=27:
        print("missing prediction from hellinger")
        
    filtered_y_test_kl = [y_t for y_t, y_p in zip(y_test_true_region, kl_divergence_pred_region) if y_p != -1]
    filtered_y_pred_kl = [y_p for y_p in kl_divergence_pred_region if y_p != -1]
    if len(filtered_y_pred_kl)!=27:
        print("missing prediction from kl")
        

    # --- Calculate and store metrics for the current region ---
    region_metrics = {}

    # Log-likelihood Metrics
    if filtered_y_pred_ll: # Ensure there are valid predictions
        region_metrics['loglikelihood'] = {
            'accuracy': accuracy_score(filtered_y_test_ll, filtered_y_pred_ll),
            'f1_macro': f1_score(filtered_y_test_ll, filtered_y_pred_ll, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_ll, filtered_y_pred_ll, labels=phases),
            'classification_report': classification_report(filtered_y_test_ll, filtered_y_pred_ll, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['loglikelihood'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}


    # Hellinger Distance Metrics
    if filtered_y_pred_h:
        region_metrics['hellinger'] = {
            'accuracy': accuracy_score(filtered_y_test_h, filtered_y_pred_h),
            'f1_macro': f1_score(filtered_y_test_h, filtered_y_pred_h, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_h, filtered_y_pred_h, labels=phases),
            'classification_report': classification_report(filtered_y_test_h, filtered_y_pred_h, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['hellinger'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    # KL-Divergence Metrics
    if filtered_y_pred_kl:
        region_metrics['kl_divergence'] = {
            'accuracy': accuracy_score(filtered_y_test_kl, filtered_y_pred_kl),
            'f1_macro': f1_score(filtered_y_test_kl, filtered_y_pred_kl, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_kl, filtered_y_pred_kl, labels=phases),
            'classification_report': classification_report(filtered_y_test_kl, filtered_y_pred_kl, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['kl_divergence'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    all_results[region] = region_metrics


# --- Display Results for All Regions ---
print("\n" + "="*50)
print("             OVERALL RESULTS SUMMARY             ")
print("="*50)

for region, metrics in all_results.items():
    print(f"\n### Region: {region} ###")

    for method, data in metrics.items():
        print(f"\n--- Method: {method.replace('_', ' ').title()} ---")
        print(f"Accuracy: {data['accuracy']:.4f}")
        print(f"F1-Macro Score: {data['f1_macro']:.4f}")

        if data['confusion_matrix'] is not None:
            print("\nConfusion Matrix:")
            cm_df = pd.DataFrame(data['confusion_matrix'],
                                 index=[f'True Phase {p}' for p in phases],
                                 columns=[f'Pred Phase {p}' for p in phases])
            print(cm_df)
        else:
            print("\nConfusion Matrix: Not available (no valid predictions)")

        if data['classification_report'] is not None:
            print("\nClassification Report:")
            # Convert classification report dict to a nice DataFrame for display
            report_df = pd.DataFrame(data['classification_report']).transpose()
            print(report_df)
        else:
            print("\nClassification Report: Not available (no valid predictions)")

    print("\n" + "-"*40)


--- Processing Region: ph1_ne ---

--- Processing Region: ph0_sw ---

--- Processing Region: ph1_nw ---

--- Processing Region: ph1_sw ---

--- Processing Region: ph2_ne ---

             OVERALL RESULTS SUMMARY             

### Region: ph1_ne ###

--- Method: Loglikelihood ---
Accuracy: 0.5926
F1-Macro Score: 0.4183

Confusion Matrix:
              Pred Phase 0  Pred Phase 1  Pred Phase 2
True Phase 0             0             4             0
True Phase 1             0            11             0
True Phase 2             0             7             5

Classification Report:
              precision    recall  f1-score    support
Phase 0        0.000000  0.000000  0.000000   4.000000
Phase 1        0.500000  1.000000  0.666667  11.000000
Phase 2        1.000000  0.416667  0.588235  12.000000
accuracy       0.592593  0.592593  0.592593   0.592593
macro avg      0.500000  0.472222  0.418301  27.000000
weighted avg   0.648148  0.592593  0.533043  27.000000

--- Method: Hellinger ---
Accu

### Method 3: training from bootstrap samples of full collection

In [9]:
# --- Parallelized LOOCV Function ---
def process_fold_full_bootstrap(train_index, test_index, X, y, n_components, phases, dim, num_subjects):
    X_train_list = [X[ind] for ind in train_index] # Keep as list of arrays
    X_test_single = np.array(X[test_index[0]]) # Test data for one subject
    y_train = y[train_index]
    y_test = y[test_index][0] # y_test is a single value

    # Prepare data for each phase model
    phase_data = {}
    for p in phases:
        # Filter X_train_list based on y_train
        current_phase_samples = [X_train_list[i] for i in range(len(X_train_list)) if y_train[i] == p]
        if current_phase_samples:
            data = np.vstack(current_phase_samples)
            n_sub = data.shape[0]
            samples = []
            for iters in range(50):
                samples.append(resample(data, n_samples=int(np.floor(n_sub/10)), replace=True))
            phase_data[p] = np.vstack(samples)
        else:
            phase_data[p] = np.array([]) # Empty array if no data for this phase

    # Fit GMMs for each phase
    models = {}
    for p_idx, p in enumerate(phases):
        if phase_data[p].shape[0] > 0:
            try:
                model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=phase_data[p]
                )
                model.fit(
                    X=phase_data[p],
                    weights=phase_data[p][:,1] - phase_data[p][:,0] ,
                    stop_threshold=.001,
                    verbose=False # Suppress verbose output during fitting
                )
                models[p] = model
            except Exception as e:
                # Handle cases where GMM fitting might fail (e.g., too few samples)
                # print(f"Warning: GMM fitting failed for phase {p} in a fold: {e}")
                models[p] = None
        else:
            models[p] = None

    # --- Prediction via loglikelihood ---
    bics = []
    for p in phases:
        ll = np.sum(models[p].log_probability(X_test_single))
        bics.append(-2*ll+math.log(phase_data[p].shape[0])*6*(p+2))
    # based on which model minimizes bic
    y_pred_l = phases[np.argmin(bics)]
    
    # --- Prepare test models for Hellinger and KL-Divergence ---
    test_models = {}
    for p_idx, p in enumerate(phases):
        if X_test_single.shape[0] > 0:
            try:
                test_model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=X_test_single
                )
                test_model.fit(
                    X=X_test_single,
                    weights=X_test_single[:,1] - X_test_single[:,0], 
                    stop_threshold=.001,
                    verbose=False
                )
                test_models[p] = test_model
            except Exception as e:
                # print(f"Warning: Test GMM fitting failed for phase {p} in a fold: {e}")
                test_models[p] = None
        else:
            test_models[p] = None

    # --- Prediction via Hellinger distance ---
    hdist = []
    for p in phases:
        if models[p] and test_models[p]:
            hdist.append(Hellinger(models[p], test_models[p], dim))
        else:
            hdist.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_h = phases[np.argmin(hdist)] if not np.all(np.isinf(hdist)) else -1

    # --- Prediction via KL-divergence ---
    kldiv = []
    for p in phases:
        if models[p] and test_models[p]:
            kldiv.append(kl_divergence(models[p], test_models[p], dim))
        else:
            kldiv.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_k = phases[np.argmin(kldiv)] if not np.all(np.isinf(kldiv)) else -1

    return y_test, y_pred_l, y_pred_h, y_pred_k

In [10]:
# --- Main execution loop for multiple regions ---
regions_to_analyze = ["ph1_ne", "ph0_sw", "ph1_nw", "ph1_sw", "ph2_ne"]
all_results = {}

for region in regions_to_analyze:
    print(f"\n--- Processing Region: {region} ---")

    X = extract_quadrant(region)
    y = np.array([0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,2,
         2,2,2,
         2,2,2,2])
    n_components = num_component_dict[region]
    num_subjects = 27 
    loocv = LeaveOneOut()
    phases = [0, 1, 2] # class labels
    dim = int(region[2]) 

    # Parallel execution for the current region
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_fold_full_bootstrap)(train_index, test_index, X, y, n_components, phases, dim, num_subjects)
        for i, (train_index, test_index) in enumerate(loocv.split(X))
    )

    # Unpack results for the current region
    y_test_true_region = [res[0] for res in fold_results]
    loglikelihood_pred_region = [res[1] for res in fold_results]
    hellinger_pred_region = [res[2] for res in fold_results]
    kl_divergence_pred_region = [res[3] for res in fold_results]

    # Filter out unclassifiable predictions for metric calculation (predictions of -1)
    # This ensures metrics are only calculated on cases where a prediction was actually made
    filtered_y_test_ll = [y_t for y_t, y_p in zip(y_test_true_region, loglikelihood_pred_region) if y_p != -1]
    filtered_y_pred_ll = [y_p for y_p in loglikelihood_pred_region if y_p != -1]
    if len(filtered_y_pred_ll)!=27:
        print("missing prediction from loglikelihood")
        
    filtered_y_test_h = [y_t for y_t, y_p in zip(y_test_true_region, hellinger_pred_region) if y_p != -1]
    filtered_y_pred_h = [y_p for y_p in hellinger_pred_region if y_p != -1]
    if len(filtered_y_pred_h)!=27:
        print("missing prediction from hellinger")
        
    filtered_y_test_kl = [y_t for y_t, y_p in zip(y_test_true_region, kl_divergence_pred_region) if y_p != -1]
    filtered_y_pred_kl = [y_p for y_p in kl_divergence_pred_region if y_p != -1]
    if len(filtered_y_pred_kl)!=27:
        print("missing prediction from kl")
        

    # --- Calculate and store metrics for the current region ---
    region_metrics = {}

    # Log-likelihood Metrics
    if filtered_y_pred_ll: # Ensure there are valid predictions
        region_metrics['loglikelihood'] = {
            'accuracy': accuracy_score(filtered_y_test_ll, filtered_y_pred_ll),
            'f1_macro': f1_score(filtered_y_test_ll, filtered_y_pred_ll, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_ll, filtered_y_pred_ll, labels=phases),
            'classification_report': classification_report(filtered_y_test_ll, filtered_y_pred_ll, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['loglikelihood'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}


    # Hellinger Distance Metrics
    if filtered_y_pred_h:
        region_metrics['hellinger'] = {
            'accuracy': accuracy_score(filtered_y_test_h, filtered_y_pred_h),
            'f1_macro': f1_score(filtered_y_test_h, filtered_y_pred_h, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_h, filtered_y_pred_h, labels=phases),
            'classification_report': classification_report(filtered_y_test_h, filtered_y_pred_h, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['hellinger'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    # KL-Divergence Metrics
    if filtered_y_pred_kl:
        region_metrics['kl_divergence'] = {
            'accuracy': accuracy_score(filtered_y_test_kl, filtered_y_pred_kl),
            'f1_macro': f1_score(filtered_y_test_kl, filtered_y_pred_kl, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_kl, filtered_y_pred_kl, labels=phases),
            'classification_report': classification_report(filtered_y_test_kl, filtered_y_pred_kl, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['kl_divergence'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    all_results[region] = region_metrics


# --- Display Results for All Regions ---
print("\n" + "="*50)
print("             OVERALL RESULTS SUMMARY             ")
print("="*50)

for region, metrics in all_results.items():
    print(f"\n### Region: {region} ###")

    for method, data in metrics.items():
        print(f"\n--- Method: {method.replace('_', ' ').title()} ---")
        print(f"Accuracy: {data['accuracy']:.4f}")
        print(f"F1-Macro Score: {data['f1_macro']:.4f}")

        if data['confusion_matrix'] is not None:
            print("\nConfusion Matrix:")
            cm_df = pd.DataFrame(data['confusion_matrix'],
                                 index=[f'True Phase {p}' for p in phases],
                                 columns=[f'Pred Phase {p}' for p in phases])
            print(cm_df)
        else:
            print("\nConfusion Matrix: Not available (no valid predictions)")

        if data['classification_report'] is not None:
            print("\nClassification Report:")
            # Convert classification report dict to a nice DataFrame for display
            report_df = pd.DataFrame(data['classification_report']).transpose()
            print(report_df)
        else:
            print("\nClassification Report: Not available (no valid predictions)")

    print("\n" + "-"*40)


--- Processing Region: ph1_ne ---

--- Processing Region: ph0_sw ---

--- Processing Region: ph1_nw ---

--- Processing Region: ph1_sw ---

--- Processing Region: ph2_ne ---

             OVERALL RESULTS SUMMARY             

### Region: ph1_ne ###

--- Method: Loglikelihood ---
Accuracy: 0.5926
F1-Macro Score: 0.4183

Confusion Matrix:
              Pred Phase 0  Pred Phase 1  Pred Phase 2
True Phase 0             0             4             0
True Phase 1             0            11             0
True Phase 2             0             7             5

Classification Report:
              precision    recall  f1-score    support
Phase 0        0.000000  0.000000  0.000000   4.000000
Phase 1        0.500000  1.000000  0.666667  11.000000
Phase 2        1.000000  0.416667  0.588235  12.000000
accuracy       0.592593  0.592593  0.592593   0.592593
macro avg      0.500000  0.472222  0.418301  27.000000
weighted avg   0.648148  0.592593  0.533043  27.000000

--- Method: Hellinger ---
Accu